# WordEmbeddings

Ziele der heutigen Übung:
- WordEmbeddins anwenden
- Semantic Similarity berechnen können

Ein guter Artikel für diese Übung ist: https://www.geeksforgeeks.org/python-word-embedding-using-word2vec/  Lese ihn durch und wende ihn für diese Übung an.

Installiere als erstes die beiden Bibliotheken ```nltk```und ```gensim```

In [ ]:
pip install nltk

In [ ]:
pip install gensim

Wir möchten in der heutigen Übung ein Modell von WordEmbeddings auf Basis eines eigenen Textes generieren. Der Text sollte eine gewisse Länge haben, ein Buch ist nicht verkehrt. Freie Bücher findet man zum Beispiel auf https://www.gutenberg.org/

Lade den Text aus dem entsprechenden File:

In [3]:
BOOK = "../data/alice_im_wunderland.txt"

text = ""
with open(BOOK, encoding='utf8') as file:
    text = file.read()

Preprocessing und explorative Datenanalyse ist nie verkehrt, denn auf Basis eines schlechten, ungepflegten Datensatzes kann kein gutes Modell trainiert werden. Ersetze auf jeden Fall Zeilenumbrüche durch Leerzeichen. Je nach Buch sind eventuell auch weitere Anpassungen notwendig. Du solltest danach einen fortlaufenden Text haben. Es empfiehlt sich auch einige Statistiken über den Text auszugeben, um ein Gefühl für dessen Eigenschaften zu bekommen (z.B. Satz-, Wort-, Characteranzahl).Die NLTK Bibliothek hat dafür  einige praktischen Funktionen, die wir nun nutzen möchten. Recherchiere hierzu in der API, welche Funktionen sinnvoll sind:: https://www.nltk.org/api/nltk.tokenize.html

In [4]:
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.stem import WordNetLemmatizer

sentences = sent_tokenize(text, language='german')
words = word_tokenize(text, language='german')

print("Dataset Exploration:")
print("\tNumber of characters: ", len(text))
print("\tNumber of words: ", len(words))
print("\tNumber of sentences: ", len(sentences))

Dataset Exploration:
	Number of characters:  179155
	Number of words:  36312
	Number of sentences:  1844


Nutze nun ```gensim.models.Word2Vec``` um ein CBOW und ein SkipGram WordEmbedding-Modell aus deinem Text zu erstellen:

In [5]:
data = []

for i in sent_tokenize(text, language='german'):
    data.append([x.lower() for x in words])

In [ ]:
data

In [ ]:
from gensim.models import Word2Vec

VECTOR_SIZE = 100
WINDOW = 3

# Create CBOW model
model_cbow = Word2Vec(data, min_count=1, vector_size=VECTOR_SIZE, window=WINDOW)
model_cbow.name = "CBOW"

# Create SkipGram model
model_skip_gram = Word2Vec(data, min_count=1, vector_size=VECTOR_SIZE, window=WINDOW, sg=1)
model_skip_gram.name = "SkipGram"

Schau dir bei beiden Modellen einige WordEmbeddings an.

In [7]:
for w in ['alice', 'wunderland', 'kaninchen', 'haus']:
    print("Word Embedding for: ", w)
    print(f"\t{model_cbow.name}: ", model_cbow.wv[w])
    print(f"\t{model_skip_gram.name}: ", model_skip_gram.wv[w], "\n")

Word Embedding for:  alice
	CBOW:  [-3.60449934e+00  4.92602587e-01  2.72921801e+00 -5.34910440e+00
  8.34595263e-01 -2.56723046e+00 -2.30294824e+00 -1.68036497e+00
  1.01590931e+00 -6.34835958e+00 -7.40144372e-01 -2.32210517e+00
  1.50192189e+00  1.95179296e+00 -1.94424999e+00 -4.37695503e+00
 -2.70839262e+00  1.38049352e+00  6.68742359e-02 -2.70230961e+00
  4.38702375e-01 -2.56407118e+00  3.17347693e+00 -2.23439097e+00
 -9.34034944e-01  3.06308556e+00 -4.96129656e+00  3.80678326e-01
 -4.33416319e+00  2.59767151e+00  5.40112925e+00 -6.34449720e-02
 -1.70474574e-01 -2.85065103e+00  6.37258450e-03  1.47536683e+00
 -3.69030714e-01 -2.63721681e+00  1.72542727e+00 -1.21798384e+00
 -3.91757035e+00 -1.07693732e-01  3.60906959e-01 -1.34351766e+00
  2.36695743e+00 -3.80549526e+00 -1.94683388e-01 -6.46090508e-01
  9.86675799e-01 -2.80762100e+00 -3.51375937e+00  1.31647599e+00
  1.29004753e+00 -2.76688743e+00 -8.05408537e-01 -3.85709190e+00
  1.79198027e+00 -2.35488963e+00 -2.21321678e+00  2.350

In der Vorlesung haben wir die Ähnlichkeit von Wörtern mit der Kosinus-Ähnlichkeit zwischen zwei WordEmbedding Vektoren berechnet. Nutze die Formel um zwischen zwei Vektoren die Ähnlichkeit zu bestimmen:

In [8]:
from numpy import dot
from numpy.linalg import norm

sim_tuple = (('alice', 'wunderland'), ('alice', 'kaninchen'), ('alice', 'haus'))

for tuple in sim_tuple:
    for model in [model_cbow, model_skip_gram]:
        vec1 = model.wv[tuple[0]]
        vec2 = model.wv[tuple[1]]
        simliarity = dot(vec1, vec2) / (norm(vec1) * norm(vec2))
        print(f"Simliarity between {tuple[0]} and {tuple[1]} ({model.name}): {simliarity}")
    

Simliarity between alice and wunderland (CBOW): 0.17133022844791412
Simliarity between alice and wunderland (SkipGram): 0.17911238968372345
Simliarity between alice and kaninchen (CBOW): 0.08448655903339386
Simliarity between alice and kaninchen (SkipGram): 0.13810627162456512
Simliarity between alice and haus (CBOW): -0.0955498144030571
Simliarity between alice and haus (SkipGram): 0.01466673705726862


Nutze nun die ```similarity``` Funktion, ist das Ergebnis gleich deiner eigenen Berechnung? Woran könnte der Unterschied liegen?

In [9]:
for tuple in sim_tuple:
    for model in [model_cbow, model_skip_gram]:
        simliarity = model.wv.similarity(tuple[0], tuple[1])
        print(f"Simliarity between {tuple[0]} and {tuple[1]} ({model.name}): {simliarity}")

Simliarity between alice and wunderland (CBOW): 0.17133024334907532
Simliarity between alice and wunderland (SkipGram): 0.17911238968372345
Simliarity between alice and kaninchen (CBOW): 0.08448656648397446
Simliarity between alice and kaninchen (SkipGram): 0.13810628652572632
Simliarity between alice and haus (CBOW): -0.0955498069524765
Simliarity between alice and haus (SkipGram): 0.014666736125946045


Interessant ist die Funktion ```most_similar``` lese dir die Doku durch und probiere einige Wörter deines Buches aus:

In [10]:
model.wv.most_similar(positive=['alice', 'wunderland', 'kaninchen'], negative=['mann'])

[('zusammen', 0.4232892692089081),
 ('lewis', 0.41793644428253174),
 ('karre', 0.4098762273788452),
 ('author', 0.3979603946208954),
 ('unentschieden', 0.3915139436721802),
 ('ausgraben', 0.3912370800971985),
 ('sache', 0.3850053548812866),
 ('13', 0.3839111328125),
 ('rufend', 0.3820779323577881),
 ('weiße', 0.3798992931842804)]